Here all the libraries and modelue calls will be written 

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

the main thing i'm followng is a pipline. whch would help me to keep my mind in check what i'm doing.

In [4]:

# i'm using a dictonary to store all the tables together as one table.

tables = {
        "train": pd.read_csv("D:/Projects/Coupen Redemption/train/train.csv"),
        "test": pd.read_csv("D:/Projects/Coupen Redemption/test.csv"),
        "campaign_data": pd.read_csv("D:/Projects/Coupen Redemption/train/campaign_data.csv"),
        "coupon_item_mapping": pd.read_csv("D:/Projects/Coupen Redemption/train/coupon_item_mapping.csv"),
        "customer_demographics": pd.read_csv("D:/Projects/Coupen Redemption/train/customer_demographics.csv"),
        "customer_transaction_data": pd.read_csv("D:/Projects/Coupen Redemption/train/customer_transaction_data.csv"),
        "item_data": pd.read_csv("D:/Projects/Coupen Redemption/train/item_data.csv"),
    }

Performing some basic checks to see how the data is

In [5]:
tables['customer_demographics']['marital_status'].unique()
tables['customer_demographics'].isnull().sum()

customer_id         0
age_range           0
marital_status    329
rented              0
family_size         0
no_of_children    538
income_bracket      0
dtype: int64

In [6]:
tables['customer_demographics']['marital_status'] = tables['customer_demographics']['marital_status'].fillna(tables['customer_demographics']['marital_status'].mode()[0])

In [7]:
tables['customer_demographics']['marital_status'].unique()

array(['Married', 'Single'], dtype=object)

one dataset problem done

In [8]:
tables['customer_demographics']['no_of_children'].unique()

array([nan, '1', '2', '3+'], dtype=object)

In [9]:
tables['customer_demographics']['no_of_children'] = tables['customer_demographics']['no_of_children'].fillna(tables['customer_demographics']['no_of_children'].mode()[0])

In [10]:
tables['customer_demographics'].isnull().sum()

customer_id       0
age_range         0
marital_status    0
rented            0
family_size       0
no_of_children    0
income_bracket    0
dtype: int64

In [11]:
tables['customer_demographics']['family_size'] = pd.to_numeric(tables['customer_demographics']['family_size'].astype(str).str.rstrip('+'), errors='coerce')
tables['customer_demographics']['no_of_children'] = pd.to_numeric(tables['customer_demographics']['no_of_children'].astype(str).str.rstrip('+'), errors='coerce')

In [12]:
print(tables['customer_demographics']['no_of_children'].unique())
print(tables['customer_demographics']['family_size'].unique())

[1 2 3]
[2 3 4 1 5]


In [13]:
tables['customer_demographics']['age_range'].unique()

array(['70+', '46-55', '26-35', '36-45', '18-25', '56-70'], dtype=object)

In [14]:
age_dummies = pd.get_dummies(tables['customer_demographics']['age_range'], prefix='age_range', dtype=int)
tables['customer_demographics'] = pd.concat([tables['customer_demographics'], age_dummies], axis=1)

In [15]:
tables['customer_demographics']

,customer_id,age_range,marital_status,rented,family_size,no_of_children,income_bracket,age_range_18-25,age_range_26-35,age_range_36-45,age_range_46-55,age_range_56-70,age_range_70+
0,1,70+,Married,0,2,1,4,0,0,0,0,0,1
1,6,46-55,Married,0,2,1,5,0,0,0,1,0,0
2,7,26-35,Married,0,3,1,3,0,1,0,0,0,0
3,8,26-35,Married,0,4,2,6,0,1,0,0,0,0
4,10,46-55,Single,0,1,1,5,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,1577,36-45,Married,0,2,1,5,0,0,1,0,0,0
756,1578,46-55,Married,0,3,1,6,0,0,0,1,0,0
757,1579,46-55,Married,0,1,1,4,0,0,0,1,0,0
758,1580,26-35,Married,0,2,1,5,0,1,0,0,0,0


One Complete data set is done

customer_transaction_data. lets go with this dataset

In [16]:
tables['customer_transaction_data'].isnull().sum()

date               0
customer_id        0
item_id            0
quantity           0
selling_price      0
other_discount     0
coupon_discount    0
dtype: int64

In [17]:
tables['customer_transaction_data']['selling_price'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 1324566 entries, 0 to 1324565
Series name: selling_price
Non-Null Count    Dtype  
--------------    -----  
1324566 non-null  float64
dtypes: float64(1)
memory usage: 10.1 MB


In [18]:
age_dummies = pd.get_dummies(tables['customer_demographics']['marital_status'], prefix='marital_status', dtype=int)
tables['customer_demographics'] = pd.concat([tables['customer_demographics'], age_dummies], axis=1)

In [19]:
age_dummies = pd.get_dummies(tables['campaign_data']['campaign_type'], prefix='campaign_type', dtype=int)
tables['campaign_data'] = pd.concat([tables['campaign_data'], age_dummies], axis=1)

In [20]:
tables['customer_transaction_data']['date'] = pd.to_datetime(tables['customer_transaction_data']['date'])
tables['campaign_data']['start_date'] = pd.to_datetime(tables['campaign_data']['start_date'], format='%d/%m/%y')
tables['campaign_data']['end_date'] = pd.to_datetime(tables['campaign_data']['end_date'], format='%d/%m/%y')
tables['campaign_data']['campaign_duration_days'] = (tables['campaign_data']['end_date'] - tables['campaign_data']['start_date']).dt.days

In [21]:
coupon_items = tables['coupon_item_mapping'].merge(tables['item_data'], on='item_id', how='left')
coupon_categories = coupon_items.groupby('coupon_id')['category'].apply(set).to_dict()
coupon_brands = coupon_items.groupby('coupon_id')['brand'].apply(set).to_dict()

In [22]:
from collections import Counter

txn_with_item = tables['customer_transaction_data'].merge(tables['item_data'], on='item_id', how='left')
txn_with_item = txn_with_item.sort_values(['customer_id', 'date'])

customer_groups = {}
for customer_id, grp in txn_with_item.groupby('customer_id', sort=False):
    customer_groups[customer_id] = {
        'dates': grp['date'].values,
        'categories': grp['category'].values,
        'brands': grp['brand'].values,
        'selling_price': grp['selling_price'].values,
        'coupon_discount': grp['coupon_discount'].values,
    }

In [23]:
cutoff_map = tables['campaign_data'].set_index('campaign_id')['start_date'].to_dict()

def build_history_cache(base_df):
    cache = {}
    pairs = base_df[['customer_id', 'campaign_id']].drop_duplicates().itertuples(index=False)
    for customer_id, campaign_id in pairs:
        cutoff = cutoff_map.get(campaign_id)
        history = customer_groups.get(customer_id)
        if history is None or cutoff is None:
            cache[(customer_id, campaign_id)] = (Counter(), Counter(), 0, 0.0, 0.0, 0.0)
            continue
        idx = np.searchsorted(history['dates'], np.datetime64(cutoff), side='left')
        past_prices = history['selling_price'][:idx]
        past_discounts = history['coupon_discount'][:idx]
        n_past = idx
        total_spend = past_prices.sum() if n_past > 0 else 0.0
        avg_spend = past_prices.mean() if n_past > 0 else 0.0
        coupon_usage_rate = (past_discounts < 0).sum() / n_past if n_past > 0 else 0.0
        coupon_discount_ratio = (-past_discounts.sum() / total_spend) if total_spend > 0 else 0.0
        cache[(customer_id, campaign_id)] = (
            Counter(history['categories'][:idx]),
            Counter(history['brands'][:idx]),
            n_past, total_spend, avg_spend, coupon_usage_rate, coupon_discount_ratio,
        )
    return cache

In [24]:
def build_features(base_df, cache):
    rows = []
    for customer_id, coupon_id, campaign_id in base_df[['customer_id', 'coupon_id', 'campaign_id']].itertuples(index=False):
        cat_counts, brand_counts, n_past, total_spend, avg_spend, usage_rate, discount_ratio = cache[(customer_id, campaign_id)]
        cats = coupon_categories.get(coupon_id, set())
        brands = coupon_brands.get(coupon_id, set())
        n_cat = sum(cat_counts[c] for c in cats)
        n_brand = sum(brand_counts[b] for b in brands)
        relevance_ratio = (n_cat + n_brand) / n_past if n_past > 0 else 0.0
        rows.append((n_cat, n_brand, relevance_ratio, n_past, total_spend, avg_spend, usage_rate, discount_ratio))
    cols = ['n_category_matches', 'n_brand_matches', 'coupon_relevance_ratio',
            'n_transactions', 'total_spend', 'avg_spend', 'coupon_usage_rate', 'coupon_discount_ratio']
    return pd.DataFrame(rows, columns=cols, index=base_df.index)

In [25]:
def assemble(split):
    base = tables[split].merge(tables['campaign_data'], on='campaign_id', how='left')
    base = base.merge(tables['customer_demographics'], on='customer_id', how='left')
    base['has_demographics'] = base['income_bracket'].notna().astype(int)

    cache = build_history_cache(base)
    feats = build_features(base, cache)
    base = pd.concat([base, feats], axis=1)
    return base

train_features = assemble('train')
test_features = assemble('test')

train_features.to_csv('D:/Projects/Coupen Redemption 2/train_features.csv', index=False)
test_features.to_csv('D:/Projects/Coupen Redemption 2/test_features.csv', index=False)
print(train_features.shape, test_features.shape)

(78369, 34) (50226, 33)


In [26]:
train_features['coupon_relevance_ratio'].describe()

count    78369.000000
mean         0.587476
std          0.343679
min          0.000000
25%          0.201226
50%          0.696803
75%          0.801907
max          1.655296
Name: coupon_relevance_ratio, dtype: float64

In [27]:
train_features.columns

Index(['id', 'campaign_id', 'coupon_id', 'customer_id', 'redemption_status',
       'campaign_type', 'start_date', 'end_date', 'campaign_type_X',
       'campaign_type_Y', 'campaign_duration_days', 'age_range',
       'marital_status', 'rented', 'family_size', 'no_of_children',
       'income_bracket', 'age_range_18-25', 'age_range_26-35',
       'age_range_36-45', 'age_range_46-55', 'age_range_56-70',
       'age_range_70+', 'marital_status_Married', 'marital_status_Single',
       'has_demographics', 'n_category_matches', 'n_brand_matches',
       'coupon_relevance_ratio', 'n_transactions', 'total_spend', 'avg_spend',
       'coupon_usage_rate', 'coupon_discount_ratio'],
      dtype='object')

Now i have completed the whole feature engineering part. now lets start the training part

In [28]:
train_features.isnull().sum()

id                            0
campaign_id                   0
coupon_id                     0
customer_id                   0
redemption_status             0
campaign_type                 0
start_date                    0
end_date                      0
campaign_type_X               0
campaign_type_Y               0
campaign_duration_days        0
age_range                 34708
marital_status            34708
rented                    34708
family_size               34708
no_of_children            34708
income_bracket            34708
age_range_18-25           34708
age_range_26-35           34708
age_range_36-45           34708
age_range_46-55           34708
age_range_56-70           34708
age_range_70+             34708
marital_status_Married    34708
marital_status_Single     34708
has_demographics              0
n_category_matches            0
n_brand_matches               0
coupon_relevance_ratio        0
n_transactions                0
total_spend                   0
avg_spen

In [29]:
train_features = pd.read_csv('D:/Projects/Coupen Redemption 2/train_features.csv')

i had some null values and strings here so i'm fixing them

In [30]:
numeric_cols_to_fill = ['family_size', 'no_of_children', 'income_bracket', 'rented']
for col in numeric_cols_to_fill:
    train_features[col] = train_features[col].fillna(train_features[col].mode()[0])

dummy_cols_to_fill = [
    'age_range_18-25', 'age_range_26-35', 'age_range_36-45',
    'age_range_46-55', 'age_range_56-70', 'age_range_70+',
    'marital_status_Married', 'marital_status_Single',
]
for col in dummy_cols_to_fill:
    train_features[col] = train_features[col].fillna(0)

In [31]:
train_features.isnull().sum()

id                            0
campaign_id                   0
coupon_id                     0
customer_id                   0
redemption_status             0
campaign_type                 0
start_date                    0
end_date                      0
campaign_type_X               0
campaign_type_Y               0
campaign_duration_days        0
age_range                 34708
marital_status            34708
rented                        0
family_size                   0
no_of_children                0
income_bracket                0
age_range_18-25               0
age_range_26-35               0
age_range_36-45               0
age_range_46-55               0
age_range_56-70               0
age_range_70+                 0
marital_status_Married        0
marital_status_Single         0
has_demographics              0
n_category_matches            0
n_brand_matches               0
coupon_relevance_ratio        0
n_transactions                0
total_spend                   0
avg_spen

In [32]:
train_features.to_csv('D:/Projects/Coupon Redemption Predictor/data/processed/train_features.csv')

In [33]:
numeric_cols_to_fill = ['family_size', 'no_of_children', 'income_bracket', 'rented']
for col in numeric_cols_to_fill:
    test_features[col] = test_features[col].fillna(test_features[col].mode()[0])

dummy_cols_to_fill = [
    'age_range_18-25', 'age_range_26-35', 'age_range_36-45',
    'age_range_46-55', 'age_range_56-70', 'age_range_70+',
    'marital_status_Married', 'marital_status_Single',
]
for col in dummy_cols_to_fill:
    test_features[col] = test_features[col].fillna(0)

In [35]:
test_features.to_csv('D:/Projects/Coupon Redemption Predictor/data/processed/train_features2.csv')